# LoRA Training Comparison and Analysis

This notebook converts the LoRA training script to Jupyter format with configurable parameters for multiple runs and comparison.

## Features:
- Configurable LoRA training parameters
- Multiple experimental runs with different settings
- Comparative analysis and visualization
- Remote server friendly execution
- Automatic result saving and comparison

---

## 1. Setup and Configuration

In [ ]:
# Define multiple training configurations for comparison
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

@dataclass
class TrainingConfig:
    name: str
    model_name: str = "openai/whisper-small"
    language: str = "zh"
    task: str = "transcribe"
    target_sr: int = 16000
    
    # LoRA parameters
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    target_modules: List[str] = None
    bias: str = "none"
    
    # Training parameters
    train_csv: str = "data/train.csv"
    dev_csv: str = "data/dev.csv"
    output_dir: str = "models/lora_experiments"
    
    batch_size: int = 16
    learning_rate: float = 1e-4
    num_epochs: int = 3
    warmup_steps: int = 500
    save_steps: int = 500
    eval_steps: int = 500
    logging_steps: int = 100
    
    # Data augmentation
    use_augmentation: bool = False
    augmentation_csv: str = "data/train_augmented.csv"
    
    # Advanced training parameters
    gradient_accumulation_steps: int = 1
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    lr_scheduler_type: str = "constant"
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    
    # Experiment tracking
    run_id: str = None
    run_name: str = None
    timestamp: str = None

def get_training_configs():
    """Get multiple training configurations for comparison"""
    return [
        TrainingConfig(
            name="baseline",
            lora_r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            batch_size=16,
            learning_rate=1e-4,
            num_epochs=3,
            use_augmentation=False
        ),
        TrainingConfig(
            name="high_rank",
            lora_r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            batch_size=16,
            learning_rate=1e-4,
            num_epochs=3,
            use_augmentation=False
        ),
        TrainingConfig(
            name="augmented",
            lora_r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            batch_size=16,
            learning_rate=1e-4,
            num_epochs=3,
            use_augmentation=True
        ),
        TrainingConfig(
            name="conservative",
            lora_r=4,
            lora_alpha=8,
            lora_dropout=0.05,
            batch_size=8,
            learning_rate=5e-5,
            num_epochs=5,
            use_augmentation=False
        ),
        TrainingConfig(
            name="aggressive",
            lora_r=32,
            lora_alpha=64,
            lora_dropout=0.2,
            batch_size=32,
            learning_rate=2e-4,
            num_epochs=2,
            use_augmentation=False
        ),
        TrainingConfig(
            name="experimental",
            lora_r=12,
            lora_alpha=24,
            lora_dropout=0.15,
            batch_size=20,
            learning_rate=3e-5,
            num_epochs=4,
            gradient_accumulation_steps=2,
            weight_decay=1e-5,
            max_grad_norm=0.5,
            lr_scheduler_type="cosine",
            adam_beta1=0.95,
            adam_beta2=0.98,
            use_augmentation=False
        ),
    ]

# Get configurations
training_configs = get_training_configs()

print(f"📋 Training Configurations: {len(training_configs)}")
for i, config in enumerate(training_configs):
    print(f"  {i+1}. {config.name}")
    print(f"     LoRA: r={config.lora_r}, alpha={config.lora_alpha}, dropout={config.lora_dropout}")
    print(f"     Training: lr={config.learning_rate}, batch={config.batch_size}, epochs={config.num_epochs}")
    print(f"     Advanced: grad_acc={config.gradient_accumulation_steps}, weight_decay={config.weight_decay}")
    print(f"     Augmentation: {config.use_augmentation}")
    print()

In [ ]:
# Import necessary libraries for LoRA training
import sys
sys.path.append('scripts')

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from opencc import OpenCC
import librosa
import numpy as np
from utils.timing_utils import initialize_timing, log_cell
from scripts.chinese_text_normalization import ChineseTextNormalizer, MANDARIN_CONFIG

cc = OpenCC("t2s")

def normalize(text: str) -> str:
    """Text normalization with Chinese support"""
    text = text.strip()
    text = cc.convert(text)
    import re
    text = re.sub(r"\s+", "", text)
    return text

def load_audio(path: str):
    """Load audio file"""
    audio, _ = librosa.load(path, sr=16000, mono=True)
    return audio.astype(np.float32)

def prepare_batch(batch: dict[str, Any], processor: WhisperProcessor):
    """Prepare batch for training"""
    # Load and process audio
    audio = load_audio(batch["audio"])
    
    # Process audio
    inputs = processor(
        audio=audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    
    # Prepare labels with Chinese normalization
    input_str = normalize(batch["text"])
    labels = processor.tokenizer(input_str).input_ids
    
    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
    }

def train_lora_model(config: TrainingConfig):
    """Train LoRA model with given configuration"""
    print(f"\n🚀 Starting LoRA Training: {config.name}")
    print(f"   LoRA: r={config.lora_r}, alpha={config.lora_alpha}")
    print(f"   Training: lr={config.learning_rate}, batch={config.batch_size}, epochs={config.num_epochs}")
    print(f"   Augmentation: {config.use_augmentation}")
    
    # Generate run ID
    if config.run_id is None:
        config.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        config.run_name = f"lora_{config.name}_{config.run_id}"
        config.timestamp = datetime.now().isoformat()
    
    # Create output directory
    os.makedirs(config.output_dir, exist_ok=True)
    
    # Load processor with Mandarin configuration
    processor = WhisperProcessor.from_pretrained(
        config.model_name, 
        language=config.language, 
        task=config.task
    )
    
    # Load base model
    model = WhisperForConditionalGeneration.from_pretrained(config.model_name)
    model.config.use_cache = False
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.target_modules or ["q_proj", "v_proj"],
        lora_dropout=config.lora_dropout,
        bias=config.bias,
    )
    
    # Apply LoRA to model
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Select training data
    train_csv = config.augmentation_csv if config.use_augmentation else config.train_csv
    
    # Load dataset
    dataset = load_dataset(
        "csv",
        data_files={
            "train": train_csv,
            "validation": config.dev_csv,
        },
    )
    
    # Prepare dataset
    dataset = dataset.map(
        lambda batch: prepare_batch(batch, processor),
        remove_columns=["audio", "text"],
    )
    
    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=config.output_dir,
        per_device_train_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        warmup_steps=config.warmup_steps,
        num_train_epochs=config.num_epochs,
        evaluation_strategy="epoch",
        save_steps=config.save_steps,
        eval_steps=config.eval_steps,
        logging_steps=config.logging_steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        push_to_hub=False,
        report_to=["none"],
        remove_unused_columns=False,
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        lr_scheduler_type=config.lr_scheduler_type,
        adam_beta1=config.adam_beta1,
        adam_beta2=config.adam_beta2,
        adam_epsilon=config.adam_epsilon,
    )
    
    # Create trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        tokenizer=processor.tokenizer,
        data_collator=lambda data: {"input_features": data["input_features"], "labels": data["labels"]},
    )
    
    # Train model
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time
    
    # Save final model
    final_output_dir = os.path.join(config.output_dir, config.run_name)
    trainer.save_model(final_output_dir)
    
    # Save training metadata
    training_metadata = {
        "run_id": config.run_id,
        "run_name": config.run_name,
        "timestamp": config.timestamp,
        "config": config.__dict__,
        "training_time_seconds": training_time,
        "final_model_path": final_output_dir,
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_params": sum(p.numel() for p in model.parameters()),
        "mandarin_config": MANDARIN_CONFIG,
    }
    
    metadata_path = os.path.join(final_output_dir, "training_metadata.json")
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(training_metadata, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Training completed in {training_time/60:.1f} minutes")
    print(f"📁 Model saved to: {final_output_dir}")
    print(f"📝 Metadata saved to: {metadata_path}")
    print(f"🇨🇳 Mandarin transcription config integrated")
    
    return training_metadata

print("🎯 LoRA training functions loaded with Chinese normalization")

## 2. Training Functions

In [ ]:
# Import necessary libraries for LoRA training
import sys
sys.path.append('scripts')

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from opencc import OpenCC
import librosa
import numpy as np
from utils.timing_utils import initialize_timing, log_cell

cc = OpenCC("t2s")

def normalize(text: str) -> str:
    """Text normalization"""
    text = text.strip()
    text = cc.convert(text)
    import re
    text = re.sub(r"\s+", "", text)
    return text

def load_audio(path: str):
    """Load audio file"""
    audio, _ = librosa.load(path, sr=16000, mono=True)
    return audio.astype(np.float32)

def prepare_batch(batch: dict[str, Any], processor: WhisperProcessor):
    """Prepare batch for training"""
    # Load and process audio
    audio = load_audio(batch["audio"])
    
    # Process audio
    inputs = processor(
        audio=audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    
    # Prepare labels
    input_str = normalize(batch["text"])
    labels = processor.tokenizer(input_str).input_ids
    
    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
    }

def train_lora_model(config: TrainingConfig):
    """Train LoRA model with given configuration"""
    print(f"\n🚀 Starting LoRA Training: {config.name}")
    print(f"   LoRA: r={config.lora_r}, alpha={config.lora_alpha}")
    print(f"   Training: lr={config.learning_rate}, batch={config.batch_size}, epochs={config.num_epochs}")
    print(f"   Augmentation: {config.use_augmentation}")
    
    # Generate run ID
    if config.run_id is None:
        config.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        config.run_name = f"lora_{config.name}_{config.run_id}"
        config.timestamp = datetime.now().isoformat()
    
    # Create output directory
    os.makedirs(config.output_dir, exist_ok=True)
    
    # Load processor
    processor = WhisperProcessor.from_pretrained(config.model_name, language=config.language, task=config.task)
    
    # Load base model
    model = WhisperForConditionalGeneration.from_pretrained(config.model_name)
    model.config.use_cache = False
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.target_modules or ["q_proj", "v_proj"],
        lora_dropout=config.lora_dropout,
        bias=config.bias,
    )
    
    # Apply LoRA to model
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Select training data
    train_csv = config.augmentation_csv if config.use_augmentation else config.train_csv
    
    # Load dataset
    dataset = load_dataset(
        "csv",
        data_files={
            "train": train_csv,
            "validation": config.dev_csv,
        },
    )
    
    # Prepare dataset
    dataset = dataset.map(
        lambda batch: prepare_batch(batch, processor),
        remove_columns=["audio", "text"],
    )
    
    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=config.output_dir,
        per_device_train_batch_size=config.batch_size,
        gradient_accumulation_steps=1,
        learning_rate=config.learning_rate,
        warmup_steps=config.warmup_steps,
        num_train_epochs=config.num_epochs,
        evaluation_strategy="epoch",
        save_steps=config.save_steps,
        eval_steps=config.eval_steps,
        logging_steps=config.logging_steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        push_to_hub=False,
        report_to=["none"],
        remove_unused_columns=False,
    )
    
    # Create trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        tokenizer=processor.tokenizer,
        data_collator=lambda data: {"input_features": data["input_features"], "labels": data["labels"]},
    )
    
    # Train model
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time
    
    # Save final model
    final_output_dir = os.path.join(config.output_dir, config.run_name)
    trainer.save_model(final_output_dir)
    
    # Save training metadata
    training_metadata = {
        "run_id": config.run_id,
        "run_name": config.run_name,
        "timestamp": config.timestamp,
        "config": asdict(config),
        "training_time_seconds": training_time,
        "final_model_path": final_output_dir,
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_params": sum(p.numel() for p in model.parameters()),
    }
    
    metadata_path = os.path.join(final_output_dir, "training_metadata.json")
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(training_metadata, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Training completed in {training_time/60:.1f} minutes")
    print(f"📁 Model saved to: {final_output_dir}")
    print(f"📝 Metadata saved to: {metadata_path}")
    
    return training_metadata

print("🎯 LoRA training functions loaded")

def analyze_training_results(results):
    """Analyze and visualize training results"""
    if not results:
        print("❌ No training results to analyze")
        return None
    
    # Convert to DataFrame for analysis
    df_data = []
    for result in results:
        if 'error' not in result:
            df_data.append({
                'run_name': result['run_name'],
                'config_name': result['config']['name'],
                'lora_r': result['config']['lora_r'],
                'lora_alpha': result['config']['lora_alpha'],
                'lora_dropout': result['config']['lora_dropout'],
                'batch_size': result['config']['batch_size'],
                'learning_rate': result['config']['learning_rate'],
                'num_epochs': result['config']['num_epochs'],
                'use_augmentation': result['config']['use_augmentation'],
                'training_time_minutes': result['training_time_seconds'] / 60,
                'trainable_params': result['trainable_params'],
                'total_params': result['total_params'],
                'param_efficiency': result['trainable_params'] / result['total_params'] * 100,
                'final_model_path': result['final_model_path'],
            })
    
    if not df_data:
        print("❌ No successful training results to analyze")
        return None
    
    df = pd.DataFrame(df_data)
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('LoRA Training Comparison Analysis', fontsize=16, fontweight='bold')
    
    # 1. Training Time Comparison
    ax1 = axes[0, 0]
    df_sorted = df.sort_values('training_time_minutes')
    bars = ax1.bar(range(len(df_sorted)), df_sorted['training_time_minutes'])
    ax1.set_title('Training Time by Configuration')
    ax1.set_ylabel('Training Time (minutes)')
    ax1.set_xticks(range(len(df_sorted)))
    ax1.set_xticklabels(df_sorted['config_name'], rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(df_sorted['training_time_minutes']):
        ax1.text(i, v + 0.1, f'{v:.1f}', ha='center', va='bottom')
    
    # 2. LoRA Parameters Impact
    ax2 = axes[0, 1]
    scatter = ax2.scatter(df['lora_r'], df['training_time_minutes'], 
                      s=100, alpha=0.7, c=df['lora_alpha'], cmap='viridis')
    ax2.set_xlabel('LoRA Rank (r)')
    ax2.set_ylabel('Training Time (minutes)')
    ax2.set_title('LoRA Rank vs Training Time')
    ax2.grid(True, alpha=0.3)
    
    # Add colorbar for alpha
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('LoRA Alpha')
    
    # 3. Parameter Efficiency
    ax3 = axes[0, 2]
    bars = ax3.bar(range(len(df)), df['param_efficiency'])
    ax3.set_title('Parameter Efficiency (Lower = Better)')
    ax3.set_ylabel('Trainable Parameters (%)')
    ax3.set_xticks(range(len(df)))
    ax3.set_xticklabels(df['config_name'], rotation=45)
    ax3.grid(True, alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(df['param_efficiency']):
        ax3.text(i, v + 0.01, f'{v:.2f}%', ha='center', va='bottom')
    
    # 4. Learning Rate vs Training Time
    ax4 = axes[1, 0]
    scatter = ax4.scatter(df['learning_rate'], df['training_time_minutes'], 
                      s=100, alpha=0.7, c=df['batch_size'], cmap='plasma')
    ax4.set_xlabel('Learning Rate')
    ax4.set_ylabel('Training Time (minutes)')
    ax4.set_title('Learning Rate vs Training Time')
    ax4.set_xscale('log')
    ax4.grid(True, alpha=0.3)
    
    # Add colorbar for batch size
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('Batch Size')
    
    # 5. Augmentation Impact
    ax5 = axes[1, 1]
    aug_vs_time = df.groupby('use_augmentation')['training_time_minutes'].mean()
    bars = ax5.bar(['No Augmentation', 'With Augmentation'], 
                  [aug_vs_time[False], aug_vs_time[True]])
    ax5.set_title('Data Augmentation Impact')
    ax5.set_ylabel('Average Training Time (minutes)')
    ax5.grid(True, alpha=0.3)
    
    # Add value labels
    for i, v in enumerate([aug_vs_time[False], aug_vs_time[True]]):
        ax5.text(i, v + 0.1, f'{v:.1f}', ha='center', va='bottom')
    
    # 6. Configuration Summary Table
    ax6 = axes[1, 2]
    ax6.axis('off')
    
    # Create summary text
    summary_text = "Training Configuration Summary:\\n\\n"
    
    # Best performance (fastest training)
    fastest = df.loc[df['training_time_minutes'].idxmin()]
    summary_text += f"🏆 Fastest: {fastest['config_name']} ({fastest['training_time_minutes']:.1f} min)\\n"
    
    # Most efficient (fewest parameters)
    most_efficient = df.loc[df['param_efficiency'].idxmin()]
    summary_text += f"⚡ Most Efficient: {most_efficient['config_name']} ({most_efficient['param_efficiency']:.2f}% params)\\n"
    
    # Highest rank
    highest_rank = df.loc[df['lora_r'].idxmax()]
    summary_text += f"🎯 Highest Rank: {highest_rank['config_name']} (r={highest_rank['lora_r']})\\n\\n"
    
    # Augmentation comparison
    aug_results = df[df['use_augmentation'] == True]
    no_aug_results = df[df['use_augmentation'] == False]
    
    if len(aug_results) > 0 and len(no_aug_results) > 0:
        aug_avg = aug_results['training_time_minutes'].mean()
        no_aug_avg = no_aug_results['training_time_minutes'].mean()
        summary_text += f"📊 Augmentation Impact: +{((aug_avg - no_aug_avg) / no_aug_avg * 100):.1f}% time\\n"
    
    summary_text += "\\nDetailed Results:\\n"
    for _, row in df.iterrows():
        summary_text += f"{row['config_name']}: {row['training_time_minutes']:.1f}min, {row['param_efficiency']:.1f}% params\\n"
    
    ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, 
              fontsize=10, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    plt.show()
    
    # Save plots
    plot_path = f"results/lora_training_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"📊 Analysis plots saved to: {plot_path}")
    
    return df, plot_path

def evaluate_trained_models(results):
    """Evaluate all trained LoRA models on test set"""
    from scripts.chinese_text_normalization import ChineseTextNormalizer, normalize_prediction_for_evaluation
    
    print("\\n🎯 Starting Model Evaluation")
    print("=" * 50)
    
    evaluation_results = []
    
    for result in results:
        if 'error' in result:
            print(f"⚠️  Skipping {result.get('config_name', 'unknown')} - training failed")
            continue
        
        config_name = result['config']['name']
        model_path = result['final_model_path']
        
        print(f"\\n🔍 Evaluating: {config_name}")
        print(f"   Model: {model_path}")
        
        try:
            # Load trained LoRA model
            processor = WhisperProcessor.from_pretrained(
                result['config']['model_name'],
                language="zh",
                task="transcribe"
            )
            
            model = WhisperForConditionalGeneration.from_pretrained(model_path)
            
            # Load test dataset
            test_dataset = load_dataset(
                "csv",
                data_files={"test": "data/test.csv"},
            )
            
            # Evaluate on test set
            predictions = []
            references = []
            
            for sample in test_dataset["test"]:
                # Load and process audio
                audio, _ = librosa.load(sample["audio"], sr=16000, mono=True)
                
                # Prepare input
                inputs = processor(
                    audio=audio,
                    sampling_rate=16000,
                    return_tensors="pt"
                )
                
                # Generate transcription with Mandarin configuration
                with torch.no_grad():
                    generated_ids = model.generate(
                        inputs.input_features,
                        max_length=448,
                        language="zh",
                        task="transcribe",
                        temperature=0.0,
                        do_sample=False,
                        num_beams=1,
                    )
                
                # Decode prediction
                prediction = processor.decode(generated_ids[0], skip_special_tokens=True)
                
                # Normalize both prediction and reference
                norm_pred, norm_ref = normalize_prediction_for_evaluation(
                    prediction, sample["text"]
                )
                
                predictions.append(norm_pred)
                references.append(norm_ref)
            
            # Calculate Chinese ASR metrics
            from scripts.chinese_text_normalization import evaluate_chinese_asr
            metrics = evaluate_chinese_asr(predictions, references)
            
            eval_result = {
                "config_name": config_name,
                "model_path": model_path,
                "cer": metrics["cer"],
                "char_errors": metrics["char_errors"],
                "char_total": metrics["char_total"],
                "test_samples": len(predictions),
                "config": result['config'],
            }
            
            evaluation_results.append(eval_result)
            
            print(f"   ✅ CER: {metrics['cer']:.4f}")
            print(f"   📊 Character errors: {metrics['char_errors']}/{metrics['char_total']}")
            print(f"   🎯 Test samples: {len(predictions)}")
            
        except Exception as e:
            print(f"   ❌ Evaluation failed: {e}")
            evaluation_results.append({
                "config_name": config_name,
                "model_path": model_path,
                "error": str(e),
                "cer": float('inf'),
                "config": result['config'],
            })
    
    return evaluation_results

print("🎯 Analysis and evaluation functions loaded")

In [ ]:
# Initialize timing system
notebook_start_time = time.time()
timer = initialize_timing("results/execution_logs/lora_training_timing.json")

# Select which configurations to run
# Options: run all, run specific, or run single
TRAINING_MODE = "all"  # Options: "all", "single", "specific"

# For single/specific mode, specify which configs
SELECTED_CONFIGS = ["baseline", "high_rank"]  # Only used if mode != "all"

print(f"🎛️  Training Mode: {TRAINING_MODE}")
if TRAINING_MODE == "single":
    print(f"📋 Selected Config: {SELECTED_CONFIGS[0]}")
elif TRAINING_MODE == "specific":
    print(f"📋 Selected Configs: {SELECTED_CONFIGS}")
else:
    print(f"📋 Running All {len(training_configs)} Configurations")

# Determine which configs to run
if TRAINING_MODE == "all":
    configs_to_run = training_configs
elif TRAINING_MODE == "single":
    configs_to_run = [config for config in training_configs if config.name == SELECTED_CONFIGS[0]]
else:  # specific
    configs_to_run = [config for config in training_configs if config.name in SELECTED_CONFIGS]

print(f"\n🚀 Starting {len(configs_to_run)} training experiments...")

# Run training experiments
all_training_results = []

for i, config in enumerate(configs_to_run):
    print(f"\n{'='*60}")
    print(f"Experiment {i+1}/{len(configs_to_run)}: {config.name}")
    
    try:
        with log_cell(f"LoRA Training - {config.name}"):
            result = train_lora_model(config)
            all_training_results.append(result)
    except Exception as e:
        print(f"❌ Training failed for {config.name}: {e}")
        # Add failed result
        failed_result = {
            "run_id": config.run_id or datetime.now().strftime("%Y%m%d_%H%M%S"),
            "run_name": config.run_name or f"lora_{config.name}_failed",
            "timestamp": datetime.now().isoformat(),
            "config": asdict(config),
            "error": str(e),
            "training_time_seconds": 0,
            "final_model_path": None,
        }
        all_training_results.append(failed_result)

print(f"\n🎉 All training experiments completed!")
print(f"📊 Total results: {len(all_training_results)}")

## 4. Training Results Analysis and Visualization

In [ ]:
def analyze_training_results(results):
    """Analyze and visualize training results"""
    if not results:
        print("❌ No training results to analyze")
        return None
    
    # Convert to DataFrame for analysis
    df_data = []
    for result in results:
        if 'error' not in result:
            df_data.append({
                'run_name': result['run_name'],
                'config_name': result['config']['name'],
                'lora_r': result['config']['lora_r'],
                'lora_alpha': result['config']['lora_alpha'],
                'lora_dropout': result['config']['lora_dropout'],
                'batch_size': result['config']['batch_size'],
                'learning_rate': result['config']['learning_rate'],
                'num_epochs': result['config']['num_epochs'],
                'use_augmentation': result['config']['use_augmentation'],
                'training_time_minutes': result['training_time_seconds'] / 60,
                'trainable_params': result['trainable_params'],
                'total_params': result['total_params'],
                'param_efficiency': result['trainable_params'] / result['total_params'] * 100,
                'final_model_path': result['final_model_path'],
            })
    
    if not df_data:
        print("❌ No successful training results to analyze")
        return None
    
    df = pd.DataFrame(df_data)
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('LoRA Training Comparison Analysis', fontsize=16, fontweight='bold')
    
    # 1. Training Time Comparison
    ax1 = axes[0, 0]
    df_sorted = df.sort_values('training_time_minutes')
    bars = ax1.bar(range(len(df_sorted)), df_sorted['training_time_minutes'])
    ax1.set_title('Training Time by Configuration')
    ax1.set_ylabel('Training Time (minutes)')
    ax1.set_xticks(range(len(df_sorted)))
    ax1.set_xticklabels(df_sorted['config_name'], rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(df_sorted['training_time_minutes']):
        ax1.text(i, v + 0.1, f'{v:.1f}', ha='center', va='bottom')
    
    # 2. LoRA Parameters Impact
    ax2 = axes[0, 1]
    scatter = ax2.scatter(df['lora_r'], df['training_time_minutes'], 
                      s=100, alpha=0.7, c=df['lora_alpha'], cmap='viridis')
    ax2.set_xlabel('LoRA Rank (r)')
    ax2.set_ylabel('Training Time (minutes)')
    ax2.set_title('LoRA Rank vs Training Time')
    ax2.grid(True, alpha=0.3)
    
    # Add colorbar for alpha
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('LoRA Alpha')
    
    # 3. Parameter Efficiency
    ax3 = axes[0, 2]
    bars = ax3.bar(range(len(df)), df['param_efficiency'])
    ax3.set_title('Parameter Efficiency (Lower = Better)')
    ax3.set_ylabel('Trainable Parameters (%)')
    ax3.set_xticks(range(len(df)))
    ax3.set_xticklabels(df['config_name'], rotation=45)
    ax3.grid(True, alpha=0.3)
    
    # 4. Learning Rate vs Training Time
    ax4 = axes[1, 0]
    scatter = ax4.scatter(df['learning_rate'], df['training_time_minutes'], 
                      s=100, alpha=0.7, c=df['batch_size'], cmap='plasma')
    ax4.set_xlabel('Learning Rate')
    ax4.set_ylabel('Training Time (minutes)')
    ax4.set_title('Learning Rate vs Training Time')
    ax4.set_xscale('log')
    ax4.grid(True, alpha=0.3)
    
    # Add colorbar for batch size
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('Batch Size')
    
    # 5. Augmentation Impact
    ax5 = axes[1, 1]
    aug_vs_time = df.groupby('use_augmentation')['training_time_minutes'].mean()
    bars = ax5.bar(['No Augmentation', 'With Augmentation'], [aug_vs_time[False], aug_vs_time[True]])
    ax5.set_title('Data Augmentation Impact')
    ax5.set_ylabel('Average Training Time (minutes)')
    ax5.grid(True, alpha=0.3)
    
    # Add value labels
    for i, v in enumerate([aug_vs_time[False], aug_vs_time[True]]):
        ax5.text(i, v + 0.1, f'{v:.1f}', ha='center', va='bottom')
    
    # 6. Configuration Summary Table
    ax6 = axes[1, 2]
    ax6.axis('off')
    
    # Create summary text
    summary_text = "Training Configuration Summary:\n\n"
    
    # Best performance (fastest training)
    fastest = df.loc[df['training_time_minutes'].idxmin()]
    summary_text += f"🏆 Fastest: {fastest['config_name']} ({fastest['training_time_minutes']:.1f} min)\n"
    
    # Most efficient (fewest parameters)
    most_efficient = df.loc[df['param_efficiency'].idxmin()]
    summary_text += f"⚡ Most Efficient: {most_efficient['config_name']} ({most_efficient['param_efficiency']:.1f}% params)\n"
    
    # Highest rank
    highest_rank = df.loc[df['lora_r'].idxmax()]
    summary_text += f"🎯 Highest Rank: {highest_rank['config_name']} (r={highest_rank['lora_r']})\n\n"
    
    # Augmentation comparison
    aug_results = df[df['use_augmentation'] == True]
    no_aug_results = df[df['use_augmentation'] == False]
    
    if len(aug_results) > 0 and len(no_aug_results) > 0:
        aug_avg = aug_results['training_time_minutes'].mean()
        no_aug_avg = no_aug_results['training_time_minutes'].mean()
        summary_text += f"📊 Augmentation Impact: +{((aug_avg - no_aug_avg) / no_aug_avg * 100):.1f}% time\n"
    
    summary_text += "\nDetailed Results:\n"
    for _, row in df.iterrows():
        summary_text += f"{row['config_name']}: {row['training_time_minutes']:.1f}min, {row['param_efficiency']:.1f}% params\n"
    
    ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, 
            fontsize=10, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    plt.show()
    
    # Save plots
    plot_path = f"results/lora_training_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"📊 Analysis plots saved to: {plot_path}")
    
    return df, plot_path

# Analyze results
if all_training_results:
    analysis_df, analysis_plot = analyze_training_results(all_training_results)
else:
    print("❌ No training results to analyze")

## 5. Export Comprehensive Report

In [ ]:
def export_comprehensive_report(results, analysis_df, plot_path):
    """Export comprehensive training comparison report"""
    if not results or analysis_df is None:
        print("❌ No data to export")
        return
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Create comprehensive report
    report = {
        "experiment_timestamp": datetime.now().isoformat(),
        "experiment_type": "LoRA Training Comparison",
        "total_experiments": len(results),
        "successful_experiments": len([r for r in results if 'error' not in r]),
        "failed_experiments": len([r for r in results if 'error' in r]),
        "training_configurations": [r['config'] for r in results if 'error' not in r],
        "training_results": results,
        "analysis_summary": {
            "fastest_config": analysis_df.loc[analysis_df['training_time_minutes'].idxmin()]['config_name'] if len(analysis_df) > 0 else None,
            "most_efficient_config": analysis_df.loc[analysis_df['param_efficiency'].idxmin()]['config_name'] if len(analysis_df) > 0 else None,
            "highest_rank_config": analysis_df.loc[analysis_df['lora_r'].idxmax()]['config_name'] if len(analysis_df) > 0 else None,
            "avg_training_time": analysis_df['training_time_minutes'].mean() if len(analysis_df) > 0 else 0,
            "augmentation_impact": "Calculated from successful runs"
        },
        "recommendations": {
            "fastest_training": "Use configuration with shortest training time for quick experiments",
            "parameter_efficiency": "Lower LoRA rank and alpha for fewer trainable parameters",
            "augmentation_strategy": "Test with and without data augmentation to see impact",
            "batch_size_optimization": "Larger batch sizes may speed up training but require more memory",
        }
    }
    
    # Save report
    report_path = f"results/lora_training_comparison_report_{timestamp}.json"
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    
    # Save CSV summary
    csv_path = f"results/lora_training_comparison_summary_{timestamp}.csv"
    analysis_df.to_csv(csv_path, index=False)
    
    # Create file list
    file_list_path = f"results/lora_training_files_{timestamp}.txt"
    with open(file_list_path, 'w') as f:
        f.write(f"Analysis Plot: {plot_path}\n")
        f.write(f"Report JSON: {report_path}\n")
        f.write(f"Summary CSV: {csv_path}\n")
        
        for result in results:
            if 'error' not in result and 'final_model_path' in result:
                f.write(f"Model: {result['final_model_path']}\n")
    
    print(f"\n📋 Comprehensive Report Generated:")
    print(f"  📄 JSON Report: {report_path}")
    print(f"  📊 CSV Summary: {csv_path}")
    print(f"  📈 Analysis Plot: {plot_path}")
    print(f"  📋 File List: {file_list_path}")
    
    # Print summary
    successful = [r for r in results if 'error' not in r]
    if successful:
        print(f"\n🎯 Key Findings:")
        print(f"  Successful experiments: {len(successful)}/{len(results)}")
        print(f"  Average training time: {analysis_df['training_time_minutes'].mean():.1f} minutes")
        print(f"  Fastest configuration: {report['analysis_summary']['fastest_config']}")
        print(f"  Most efficient: {report['analysis_summary']['most_efficient_config']}")
    
    return report_path, csv_path, file_list_path

# Export comprehensive report
if all_training_results and 'analysis_df' in locals():
    report_files = export_comprehensive_report(all_training_results, analysis_df, analysis_plot)
else:
    print("❌ Insufficient data for report export")

## 6. Final Completion and Summary

### Training Modes:

#### **All Mode** (Recommended):
```python
TRAINING_MODE = "all"
```
- Runs all 4 predefined configurations
- Best for comprehensive comparison
- Takes longer but provides complete analysis

#### **Single Mode**:
```python
TRAINING_MODE = "single"
SELECTED_CONFIGS = ["baseline"]
```
- Runs only specified configuration
- Good for testing specific parameters
- Faster execution

#### **Specific Mode**:
```python
TRAINING_MODE = "specific"
SELECTED_CONFIGS = ["baseline", "high_rank"]
```
- Runs multiple selected configurations
- Good for targeted experiments

### Configuration Options:

| Config | LoRA Rank | Alpha | Dropout | Batch | LR | Epochs | Augmentation |
|--------|------------|-------|--------|-------|----|--------|--------------|
| baseline | 8 | 16 | 0.1 | 16 | 1e-4 | 3 | No |
| high_rank | 16 | 32 | 0.1 | 16 | 1e-4 | 3 | No |
| augmented | 8 | 16 | 0.1 | 16 | 1e-4 | 3 | Yes |
| conservative | 4 | 8 | 0.05 | 8 | 5e-5 | 5 | No |
| aggressive | 32 | 64 | 0.2 | 32 | 2e-4 | 2 | No |

### Expected Outputs:

- **Trained Models**: `models/lora_experiments/lora_{config}_{timestamp}/`
- **Training Metadata**: `training_metadata.json` in each model directory
- **Comparison Plots**: `results/lora_training_comparison_*.png`
- **Analysis Report**: `results/lora_training_comparison_report_*.json`
- **Summary CSV**: `results/lora_training_comparison_summary_*.csv`
- **File List**: `results/lora_training_files_*.txt`

### Usage on Remote Server:

1. **Upload Data**: Ensure `data/train.csv` and `data/dev.csv` exist
2. **Configure Mode**: Set `TRAINING_MODE` and `SELECTED_CONFIGS` as needed
3. **Run All Cells**: Execute notebook sequentially
4. **Download Results**: All outputs are timestamped for easy download

---

**🎯 Ready for comprehensive LoRA training comparison on remote server!**